In [522]:
import numpy as np
import pandas as pd
import gurobipy as gp

# read xlsx
path = 'data.xlsx'
num_pieces_df = pd.read_excel(path, sheet_name='Pieces', header=None)
board_df = pd.read_excel(path, sheet_name='Board', header=None)

In [ ]:
# keep this cell collapsed to not show lol
def secret_tweaks(model: gp.Model):
    model.setParam('OutputFlag', 0)
    model.setParam('Presolve', 0)
    model.setParam('PoolSolutions', 1) 
    model.setParam('MIPFocus', 1)  # Focus on finding feasible solutions quickly
    model.setParam('TimeLimit', 3600)  # Solve for a maximum of 3600 seconds (adjust as needed)
    model.setParam('MIPGap', 0.01)  # Set a 1% optimality gap tolerance
    model.setParam('Threads', 16)  # Use 4 threads for parallel solving
    model.setParam('OutputFlag', 1)

In [517]:
from numpy import rot90, fliplr

pieces = []
piece_single = np.array([
    [1]
])
piece_double = np.array([
    [1, 1],
    [1, 1]
])
pieces.extend([piece_single, piece_double])

pieces.extend([np.repeat(piece_single, 2, axis=1), np.repeat(piece_single, 3, axis=1), np.repeat(piece_single, 4, axis=1)])
pieces.extend([rot90(np.repeat(piece_single, 2, axis=1)), rot90(np.repeat(piece_single, 3, axis=1)), rot90(np.repeat(piece_single, 4, axis=1))])

piece_l = np.array([
    [1, 0],
    [1, 1]
])
pieces.extend([piece_l, rot90(piece_l), rot90(piece_l, 3), rot90(piece_l, 2)])

piece_L = np.array([
    [1, 0, 0],
    [1, 1, 1]
])
pieces.extend([piece_L, fliplr(piece_L), rot90(fliplr(piece_L), 2), rot90(piece_L, 2)])
pieces.extend([rot90(fliplr(piece_L), 3), rot90(piece_L), rot90(piece_L, 3), rot90(fliplr(piece_L))])

piece_S = np.array([
    [0, 1, 1],
    [1, 1, 0]
])
pieces.extend([piece_S, fliplr(piece_S), rot90(piece_S, 1), rot90(fliplr(piece_S))])

piece_T = np.array([
    [0, 1, 0],
    [1, 1, 1]
])
pieces.extend([piece_T, rot90(piece_T, 2), rot90(piece_T, 3), rot90(piece_T, 1)])
counts = num_pieces_df.iloc[:len(pieces), 0].astype(int).values
available_pieces = [item for sublist in [[piece] * count for piece, count in zip(pieces, counts)] for item in sublist]

In [554]:
board = board_df.iloc[1:].fillna(0).values
expected_board_size = (board_df.iloc[0, 0], board_df.iloc[0, 1])
# board = np.where(board == 1, 0, 1)
if board.shape != expected_board_size:
    board = np.pad(board, ((0, int(expected_board_size[0] - board.shape[0])), (0, int(expected_board_size[1] - board.shape[1]))))
board

array([[1., 0., 1., 1., 0., 1., 1.],
       [0., 0., 0., 0., 0., 0., 0.],
       [1., 1., 0., 1., 1., 0., 1.],
       [1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0.],
       [1., 0., 1., 0., 0., 0., 1.],
       [0., 1., 0., 0., 1., 0., 0.]])

In [440]:
# available_pieces = []

# pattern_1 = np.array([
#     [1],
# ])

# for i in range(400): available_pieces.append(pattern_1)
# pattern_2 = np.array([
#     [1, 1],
#     [1, 0], 
#     [1, 0],
# ])
# for i in range(50): available_pieces.append(pattern_2)

In [531]:
board = np.array([
    ([0]*20),
])
board = np.zeros((20, 20))

In [503]:
# import time
# model = gp.Model("doodleFit")

# st = time.time()
# placements = np.ndarray(shape=(len(available_pieces), board.shape[0], board.shape[1]), dtype=object)
# for i, pattern in enumerate(available_pieces):
#     for row in range(board.shape[0] - len(pattern) + 1):
#         for col in range(board.shape[1] - len(pattern[0]) + 1):
#             if ~np.any((board[row:row + len(pattern), col:col + len(pattern[0])] == 1) & (pattern == 1)):
#                 placements[i][row][col] = model.addVar(vtype=gp.GRB.BINARY, name=f'pattern_{i}_{row}_{col}')
# print(time.time() - st)

# st = time.time()
# for i, pattern in enumerate(available_pieces):
#     model.addConstr(gp.quicksum(placements[i][row][col]
#                     for row in range(board.shape[0]) # TODO look at the range of row and col
#                     for col in range(board.shape[1])
#                     if placements[i][row][col]) <= 1, 
#                     f'one_placement_piece_{i}')
# print(time.time() - st)
    
# st = time.time()
# for i in range(board.shape[0]):  # Iterate through rows
#     for j in range(board.shape[1]):  # Iterate through columns
#         terms = []
#         terms.append(board[i][j])
#         for p in range(len(available_pieces)):
#             for ii in range(len(available_pieces[p])):  # Iterate through pattern rows
#                 for jj in range(len(available_pieces[p][0])):  # Iterate through pattern columns
#                     if available_pieces[p][ii][jj] == 1:
#                         if i - ii >= 0 and j - jj >= 0 and placements[p][i - ii][j - jj] is not None:
#                             terms.append(placements[p][i - ii][j - jj])
#         # Add constraint: ensure no overlap at each position on the board
#         model.addConstr(gp.quicksum(terms) <= 1, f'no_overlap_{i}_{j}')
# print(time.time() - st)

# st = time.time()
# model.setObjective(gp.quicksum(placements[p][i][j]*np.count_nonzero(available_pieces[p]) for p in range(len(available_pieces)) for i in range(board.shape[0]) for j in range(board.shape[1]) if placements[p][i][j] is not None), gp.GRB.MAXIMIZE)
# print(time.time() - st)
        
# st = time.time()
# model.update()
# print(time.time() - st)
# secret_tweaks(model)
# st = time.time()
# model.optimize()
# print(time.time() - st)

In [558]:
import time
st = time.time()
model = gp.Model("doodleFit")

st2= time.time()
placements = np.full((len(available_pieces), board.shape[0], board.shape[1]), None, dtype=object);[[[placements.__setitem__((i, row, col), model.addVar(vtype=gp.GRB.BINARY, name=f'pattern_{i}_{row}_{col}')) if ~np.any((board[row:row + len(pattern), col:col + len(pattern[0])] == 1) & (pattern == 1)) else None for col in range(board.shape[1] - len(pattern[0]) + 1)] for row in range(board.shape[0] - len(pattern) + 1)] for i, pattern in enumerate(available_pieces)]
print(time.time()-st2)

constraints = [model.addConstr(gp.quicksum([board[i][j]] + [placements[p][i - ii][j - jj] for p in range(len(available_pieces)) for ii in range(len(available_pieces[p])) for jj in range(len(available_pieces[p][0])) if available_pieces[p][ii][jj] == 1 and i - ii >= 0 and j - jj >= 0 and placements[p][i - ii][j - jj] is not None]) <= 1, f'no_overlap_{i}_{j}') for i in range(board.shape[0]) for j in range(board.shape[1])] + [model.addConstr(gp.quicksum(placements[i][row][col] for row in range(board.shape[0]) for col in range(board.shape[1]) if placements[i][row][col]) <= 1, f'one_placement_piece_{i}') for i, pattern in enumerate(available_pieces)]
# if all available_pieces have to be used below
# constraints = [model.addConstr(gp.quicksum([board[i][j]] + [placements[p][i - ii][j - jj] for p in range(len(available_pieces)) for ii in range(len(available_pieces[p])) for jj in range(len(available_pieces[p][0])) if available_pieces[p][ii][jj] == 1 and i - ii >= 0 and j - jj >= 0 and placements[p][i - ii][j - jj] is not None]) == 1, f'no_overlap_{i}_{j}') for i in range(board.shape[0]) for j in range(board.shape[1])] + [model.addConstr(gp.quicksum(placements[i][row][col] for row in range(board.shape[0]) for col in range(board.shape[1]) if placements[i][row][col]) == 1, f'one_placement_piece_{i}') for i, pattern in enumerate(available_pieces)]

pattern_counts = np.array([np.count_nonzero(pattern) for pattern in available_pieces])
model.setObjective(gp.quicksum(placements[p][i][j]*pattern_counts[p] for p in range(len(available_pieces)) for i in range(board.shape[0]) for j in range(board.shape[1]) if placements[p][i][j] is not None)+np.count_nonzero(board), gp.GRB.MAXIMIZE)

print(time.time()-st)
model.update()
secret_tweaks(model)
model.optimize()

0.5034239292144775
0.9149260520935059
Set parameter OutputFlag to value 1
Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: 12th Gen Intel(R) Core(TM) i5-12500H, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 655 rows, 8929 columns and 24676 nonzeros
Model fingerprint: 0xe7c36f8b
Variable types: 0 continuous, 8929 integer (8929 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 49.0000000
Variable types: 0 continuous, 8929 integer (8929 binary)

Root relaxation: cutoff, 185 iterations, 0.01 seconds (0.01 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0     cutoff    0        49.0000

In [ ]:
# return which pieces are placed by the model

In [ ]:
# print("\nThe optimal solutions:")
if model.status == gp.GRB.INFEASIBLE:
    print('The model is infeasible; computing IIS')
    model.computeIIS()
    for c in model.getConstrs():
        if c.IISConstr:
            print('%s' % c.constrName)
if model.status == gp.GRB.OPTIMAL:
    for var in model.getVars():
        print(f"{var.VarName}: {var.X}")

# print(f"The optimal number of courses to take is:{model.objVal}")
# list_of_variables_defined_above = [total_sold, total_quality, expected_total_quality, rev_a, rev_b, rev_c, total_rev, prod_a, prod_b, used_a, used_b, unused_a, unused_b, total_cost, total_profit]
# list_of_variables_defined_above = [unused_a, unused_b]
# for x in list_of_variables_defined_above:
#     print(f"{x}: {x.getValue()}")
# for constr in model.getConstrs():
#     print(f"Constraint: {constr.ConstrName}, Dual Value: {constr.Pi}")

In [560]:
def get_placed_pieces(placements, available_pieces):
    placed_pieces = []
    for i, piece in enumerate(available_pieces):
        for row in range(placements.shape[1]):
            for col in range(placements.shape[2]):
                if placements[i][row][col] is not None and placements[i][row][col].X == 1:
                    placed_pieces.append((piece, (row, col)))
    return placed_pieces

# After model.optimize()
placed_pieces = get_placed_pieces(placements, available_pieces)

In [561]:
placed_pieces

[(array([[1]]), (0, 1)),
 (array([[1]]), (0, 4)),
 (array([[1]]), (1, 0)),
 (array([[1]]), (1, 1)),
 (array([[1]]), (1, 2)),
 (array([[1]]), (1, 3)),
 (array([[1]]), (1, 4)),
 (array([[1]]), (1, 5)),
 (array([[1]]), (1, 6)),
 (array([[1]]), (2, 2)),
 (array([[1]]), (2, 5)),
 (array([[1]]), (3, 1)),
 (array([[1]]), (3, 2)),
 (array([[1]]), (3, 3)),
 (array([[1]]), (3, 4)),
 (array([[1]]), (3, 5)),
 (array([[1]]), (3, 6)),
 (array([[1]]), (4, 0)),
 (array([[1]]), (4, 1)),
 (array([[1]]), (4, 2)),
 (array([[1]]), (4, 4)),
 (array([[1]]), (4, 5)),
 (array([[1]]), (4, 6)),
 (array([[1]]), (5, 1)),
 (array([[1]]), (5, 3)),
 (array([[1]]), (5, 4)),
 (array([[1]]), (5, 5)),
 (array([[1]]), (6, 0)),
 (array([[1]]), (6, 2)),
 (array([[1]]), (6, 3)),
 (array([[1]]), (6, 5)),
 (array([[1]]), (6, 6))]

In [ ]:
for c in model.getConstrs():
  # print what the constraint is evaluated to
  if c.Slack < 1e-6:
    print('Constraint %s is active at solution point' % (c.ConstrName))

# IP MODEL
## Decision Variables:
Let $x_{i, j, k}$ be a binary variable denoting whether piece $k$ is placed with its top-left corner at position $(i, j)$ on the board.

## Objective Function:
Maximize the total number of covered cells:
$$
\text{Maximize} \quad \sum_{i=1}^{m} \sum_{j=1}^{n} \sum_{k=1}^{N} x_{i, j, k} \cdot \text{pattern\_counts}[k]
$$

## Constraints:
1. Ensure no overlaps between pieces and the board:
$$
\text{for each cell }(i,j):\text{board}[i][j] + \sum_{k=1}^{N} \sum_{a=0}^{R(k)-1} \sum_{b=0}^{C(k)-1} x_{i-a, j-b, k} \leq 1
$$
2. Ensure each piece is placed at most once:
$$\text{for each piece } k:\sum_{i=1}^{m} \sum_{j=1}^{n} x_{i, j, k} \leq 1 $$

## Notes:
- $m$ and $n$ are the dimensions of the board.
- $N$ is the number of available pieces.
- $R(k)$ and $C(k)$ are the number of rows and columns in piece $k$, respectively.
- $\text{pattern\_counts}[k]$ represents the count of non-zero elements in piece $k$.
- $\text{count\_nonzero(board)}$ counts the number of non-zero elements in the initial board configuration.

### Write output to excel